# 4.6 Feature Scaling

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.06_feature-scaling.ipynb)

Companion code for Section 4.6 of the book. Run the cells in order from top to bottom.

**Example 4.6.1**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import MaxAbsScaler, QuantileTransformer, PowerTransformer

np.random.seed(42)
# Dataset with very different feature scales
df = pd.DataFrame({
    'age':    np.random.normal(40, 12, 200).clip(18, 80),    # Range ~18-80
    'income': np.random.exponential(60000, 200),              # Range ~0-500k, skewed
    'score':  np.random.normal(0.5, 0.1, 200).clip(0, 1),   # Range ~0-1
    'clicks': np.random.poisson(100, 200),                    # Range ~50-200
})

print('Before scaling:')
print(df.describe().round(2))

# ── StandardScaler: (x - mean) / std ─────────────────────────────
std_scaler = StandardScaler()
X_std = std_scaler.fit_transform(df)
df_std = pd.DataFrame(X_std, columns=df.columns)
print('\nAfter StandardScaler:')
print(df_std.describe().round(4))
print(f'Mean ≈ 0: {df_std.mean().round(4).tolist()}')
print(f'Std  ≈ 1: {df_std.std().round(4).tolist()}')

# Inverse transform to get original values
X_original = std_scaler.inverse_transform(X_std)
print(f'\nInverse transform error: {np.max(np.abs(X_original - df.values)):.2e}')

**Example 4.6.2**

In [ ]:
# ── MinMaxScaler: (x - min) / (max - min) → [0, 1] ──────────────
mm_scaler = MinMaxScaler(feature_range=(0, 1))
X_mm = mm_scaler.fit_transform(df)
df_mm = pd.DataFrame(X_mm, columns=df.columns)
print('After MinMaxScaler (range=[0,1]):')
print(f'Min values: {df_mm.min().round(4).tolist()}')
print(f'Max values: {df_mm.max().round(4).tolist()}')

# ── RobustScaler: uses median and IQR - outlier resistant ─────────
# x_scaled = (x - median) / IQR
# Best when data has significant outliers
rob_scaler = RobustScaler(quantile_range=(25, 75))
X_rob = rob_scaler.fit_transform(df)
df_rob = pd.DataFrame(X_rob, columns=df.columns)
print('\nAfter RobustScaler (quantile_range=(25,75)):')
print(f'Income std before: {df.income.std():,.0f}')
print(f'Income std after:  {df_rob.income.std():.4f}')

# ── Power Transform (Yeo-Johnson) - normalise skewed distributions
pow_scaler = PowerTransformer(method='yeo-johnson')
X_pow = pow_scaler.fit_transform(df)
df_pow = pd.DataFrame(X_pow, columns=df.columns)

from scipy import stats
print('\nSkewness comparison (income - highly right-skewed):')
print(f'  Before:           {stats.skew(df.income):+.4f}')
print(f'  After StandardScaler:   {stats.skew(df_std.income):+.4f}  (no change in shape)')
print(f'  After PowerTransform:   {stats.skew(df_pow[:,1]):+.4f}  (much more normal!)')

# ── QuantileTransformer - maps to uniform or normal distribution ──
qt_scaler = QuantileTransformer(output_distribution='normal', random_state=42)
X_qt = qt_scaler.fit_transform(df)
print(f'\nAfter QuantileTransformer (→ normal):')
print(f'  income skewness: {stats.skew(X_qt[:,1]):+.4f}')

# ── Side-by-side comparison ───────────────────────────────────────
comparison = pd.DataFrame({
    'Original':     [df.income.mean(),  df.income.std(),  df.income.min(),  df.income.max()],
    'StandardScaler':[df_std.income.mean(),df_std.income.std(), df_std.income.min(), df_std.income.max()],
    'MinMaxScaler': [df_mm.income.mean(), df_mm.income.std(), df_mm.income.min(), df_mm.income.max()],
    'RobustScaler': [df_rob.income.mean(),df_rob.income.std(),df_rob.income.min(),df_rob.income.max()],
}, index=['mean','std','min','max']).round(4)
print('\nScaling method comparison for income:')
print(comparison)